# Train/Val/Test Split Integrity Audit


This audit requires the raw `L1_events_v3.parquet` event file, which is excluded from the public release per the StatsBomb academic licence ([`../data/DATA_AVAILABILITY.md`](../data/DATA_AVAILABILITY.md)). Holders of an academic licence can regenerate the input via `../scripts/cache/prepare_l1_events.py`. The pre-computed tables in Sections 3 and 4 below are reported as Tables S2 and S3 in `manuscript/supplementary.tex`.


## 1. Motivation

A common silent failure in soccer-event ML pipelines is data leakage across the train/val/test split, typically through:
- **Match overlap**: rows from one match split between train and val
- **Season miscoded**: TEST_SEASONS rows accidentally pulled into train
- **Event-id duplicates**: the same event row sampled twice
- **Non-deterministic split**: different random seeds → different splits → silent reporting drift across runs

This notebook verifies that none of these failures affect the U1 Unified split implemented in `scripts/training/train_unified.py` (lines 325–336).


## 2. Method: 8 checks

| ID | Check | Failure mode | Verdict |
|---|---|---|---|
| C1 | Match-id disjointness (train ∩ val, (train+val) ∩ test) | per-row split | PASS |
| C2 | Season composition (train+val ⊂ TRAIN_SEASONS, test ⊂ TEST_SEASONS) | season miscoded | PASS |
| C3 | Sample count per task per split (informational) | n/a | recorded |
| C4 | Class balance drift per split (informational) | distribution shift | recorded |
| C5 | Event-id uniqueness within and across splits | duplicate sampling | PASS |
| C6 | Player overlap (informational; per-match split allows it) | n/a | recorded |
| C7 | Test seasons strictly chronologically after train | temporal mix | PASS |
| C8 | Seed-42 produces deterministic split | non-determinism | PASS |

All five hard checks return PASS.

## 3. C3: Sample counts per task per split

Pre-computed from `scripts/audit/audit_split_integrity.py`:

| Task | Train | Val | Test |
|---|---:|---:|---:|
| pass | 1,181,919 | 295,960 | 735,907 |
| dribble | 33,825 | 8,558 | 21,926 |
| ball_receipt | 1,126,488 | 281,230 | 674,917 |
| shot | 30,876 | 7,640 | 18,788 |
| duel | 82,403 | 20,788 | 49,829 |
| interception | 23,909 | 5,831 | 12,085 |
| ball_recovery | 114,161 | 28,497 | 69,800 |
| pressure | 386,375 | 97,892 | 268,964 |

Train:val ratio holds at ~4:1 for every task, confirming the 80/20 per-match split partition is faithful.

## 4. C4: Class balance drift per task per split

| Task | Train | Val | Test | train-val drift | train-test drift |
|---|---:|---:|---:|---:|---:|
| pass | 0.8058 | 0.8042 | 0.8100 | 0.0016 | 0.0042 |
| dribble | 0.5236 | 0.5340 | 0.5206 | 0.0104 | 0.0031 |
| **ball_receipt** | **0.8451** | **0.8459** | **0.8832** | 0.0008 | **0.0381** |
| shot | 0.0994 | 0.0961 | 0.0998 | 0.0033 | 0.0004 |
| duel | 0.2887 | 0.2857 | 0.3062 | 0.0030 | 0.0175 |
| interception | 0.5953 | 0.5939 | 0.6094 | 0.0014 | 0.0142 |
| ball_recovery | 0.9233 | 0.9217 | 0.9245 | 0.0016 | 0.0012 |

All $|\mathrm{train{-}val}|$ drifts < 0.012 (within sampling noise). All $|\mathrm{train{-}test}|$ drifts < 0.018 except Ball Receipt at 0.038, consistent with the paper §6 covariate-shift narrative (StatsBomb 360 pipeline change at the 23/24→24/25 boundary).

## 5. Re-run the split locally

The cell below reproduces the train/val/test masks with seed 42 and asserts the C1, C5, C7, C8 invariants. Full audit is at `scripts/audit/audit_split_integrity.py` (~30 sec).

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
CACHE = PROJECT_ROOT / 'data'
PARQUET = CACHE / 'L1_events_v3.parquet'

if not PARQUET.exists():
    print(f'Skipped: {PARQUET.name} is excluded from the public release per the StatsBomb academic licence.')
    print(f'Pre-computed split-integrity results are in Sections 3 and 4 above.')
else:
    FULL_SEASONS = [
        '2_235_2022_23', '2_281_2023_24', '2_317_2024_25',
        '11_235_2022_23', '11_281_2023_24', '11_317_2024_25',
    ]
    TRAIN_SEASONS = ['2_235_2022_23', '2_281_2023_24',
                     '11_235_2022_23', '11_281_2023_24']
    TEST_SEASONS = ['2_317_2024_25', '11_317_2024_25']

    df = pd.read_parquet(PARQUET)
    df = df[df['season_dir'].isin(FULL_SEASONS)].reset_index(drop=True)

    def build_split(df, seed=42):
        np.random.seed(seed)
        is_train_season = np.isin(df['season_dir'].values, TRAIN_SEASONS)
        is_test_season = np.isin(df['season_dir'].values, TEST_SEASONS)
        train_matches = np.unique(df['match_id'].values[is_train_season])
        np.random.shuffle(train_matches)
        n_t = int(0.8 * len(train_matches))
        tr_ids = train_matches[:n_t]
        va_ids = train_matches[n_t:]
        return (np.isin(df['match_id'].values, tr_ids),
                np.isin(df['match_id'].values, va_ids),
                is_test_season, tr_ids, va_ids)

    tr, va, te, tr_ids, va_ids = build_split(df)
    te_ids = np.unique(df['match_id'].values[te])

    # C1: match disjoint
    assert len(np.intersect1d(tr_ids, va_ids)) == 0
    assert len(np.intersect1d(np.union1d(tr_ids, va_ids), te_ids)) == 0
    print('C1 match-id disjoint        OK')

    # C5: event-id uniqueness
    eids = df['event_id'].values
    assert len(np.unique(eids[tr])) == tr.sum()
    assert len(np.unique(eids[va])) == va.sum()
    assert len(np.unique(eids[te])) == te.sum()
    print('C5 event-id uniqueness      OK')

    # C7: chronology
    year = lambda s: int(s.split('_')[-2])
    assert min(year(s) for s in TEST_SEASONS) > max(year(s) for s in TRAIN_SEASONS)
    print('C7 chronology (test > train) OK')

    # C8: determinism
    tr2, va2, te2, _, _ = build_split(df, seed=42)
    assert np.array_equal(tr, tr2)
    assert np.array_equal(va, va2)
    assert np.array_equal(te, te2)
    print('C8 seed-42 determinism      OK')


Skipped: L1_events_v3.parquet is excluded from the public release per the StatsBomb academic licence.
Pre-computed split-integrity results are in Sections 3 and 4 above.


## 6. Conclusion

The U1 split has no structural leakage: all five hard checks (C1, C2, C5, C7, C8) return PASS. The only informational flag is the Ball Receipt pos_rate drift of 0.038 between train and test, which is expected and documented in Section 6 of the main paper as a consequence of the 23/24-24/25 StatsBomb pipeline change. The full static report is `../scripts/audit/audit_split_integrity.py` output, summarised as Tables S2 and S3 in the supplementary material.
